## Final Project Phase 3: Multi-Turn Peloton AI Agents with LangGraph
### Shishir Deshpande, MSDS 442

I am continuing the work I did in Phase 2 and will be implementing **three multi-turn user stories** for each of **three selected agents**. This will amount to 9 fully functional multi-turn conversations. The agents I have chosen are as follows:

1. **Business/Marketing AI Agent**
2. **Membership/Fraud Detection AI Agent**
3. **Order/Shipping AI Agent**

For my implementation, I will ensure that each conversation carries state across turns using LangGraph's `MemorySaver` checkpointer. But I will also generate a unique `thread_id` for every user story so I can prevent checkpoint bleed across scenarios.

In [ ]:
# Importing required libraries
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

from dotenv import load_dotenv
load_dotenv()
os.environ["USER_AGENT"] = "MSDS442-FinalProjectPhase3-Deshpande"

import uuid
import pandas as pd
from typing import TypedDict, Annotated, List, Literal

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
import chromadb
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from collections import Counter

from IPython.display import display, Markdown, Image

model = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)
print("All imports OK. Selected model:", model.model_name)

### Loading data and filtering to the 3 selected agents

In [ ]:
df = pd.read_csv("Requirement 4_TrainingTestingData.csv", encoding="utf-8-sig")

SELECTED_AGENTS = [
    "Business/Marketing AI Agent",
    "Membership/Fraud Detection AI Agent",
    "Order/Shipping AI Agent",
]

df = df[df["AI Agent"].isin(SELECTED_AGENTS)].reset_index(drop = True)

# Mapping short-keys for each agent
agent_key_map = {
    "Business/Marketing AI Agent": "business",
    "Membership/Fraud Detection AI Agent": "membership",
    "Order/Shipping AI Agent": "order",
}

print(f"Filtered dataset has {len(df)} rows across {df['AI Agent'].nunique()} agents")
print()
print(df.groupby(["AI Agent", "Data Type"]).size().unstack(fill_value=0))

### Building knowledge bases for each agent

In [ ]:
agent_documents = {key: [] for key in agent_key_map.values()}

for _, row in df.iterrows():
    if row["Data Type"].strip() != "Training":
        continue
    key = agent_key_map[row["AI Agent"]]
    content = f"Query: {row['Sample Query']}\nAnswer: {row['Expected Response']}"
    agent_documents[key].append(
        Document(
            page_content=content,
            metadata={"user_story":row['User-Story'], "data_type":row["Data Type"]}
        )
    )

for key, docs in agent_documents.items():
    print(f"{key}: {len(docs)} documents.")

#### Setting up the vectorstore for each agent

As validated in Phase 2, I will use `delete_collection` to prevent duplication

In [ ]:
embeddings = OpenAIEmbeddings()
chroma_client = chromadb.Client()
agent_vectorstores = {}

for key, docs in agent_documents.items():
    collection_name = f"peloton_{key}"
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=collection_name,
        client=chroma_client
    )
    agent_vectorstores[key] = vectorstore

    actual_count = vectorstore._collection.count()
    expected_count = len(docs)
    status = "OK" if actual_count == expected_count else "ERROR"
    print(f"Collection {collection_name}: {actual_count}/{expected_count} documents - {status}")

#### Multi-turn agent nodes

I will build a state that carries a list of messages using LangGraph's `add_messages` for automating appending. This will enable conversation history persistence across turns. 

I will reuse the router from Phase 2 but limit to the three agents. 

In Phase 1, Section 4, my architecture had prescribed different implementation patterns for each user story (pure RAG in some cases, RAG + tool calls in others etc.). To align Phase 3 with those designs, I will implement 3 of the 9 user stories (1 per agent) as tool calls along with RAG context. 

|Agent|User Story|Phase 1 architecture|
|---|---|---|
|Business/Marketing|Summer Fitness Challenge Metrics|Tool:`get_campaign_metrics()`|
|Membership/Fraud Detection|Password Reset|Tool: `init_password_reset`|
|Order/Shipping|Delayed order status|Tool: `get_order_status()`|

For the other 6 stories, I will use a pure RAG-based approach. I will simulate tool returns with static synthetic data, but in real production environments, the tool calls would directly call Peloton's internal marketing analytics, identity management, and order management systems, respectively.

In [ ]:
# Mock tool calls for Phase 1-aligned architecture
@tool
def get_campaign_metrics(campaign_name: str) -> str:
    """Retrieve performance metrics for a Peloton marketing campaign by its full name.
    Returns participation rate, engagement lift, and impact to subscription."""

    # Simulated responses - in a real system, this would be a database/API call
    return(
        f"Campaign: {campaign_name}\n"
        f"Participation rate: 62% (below 75% benchmark)\n"
        f"Member engagement lift: 236 (+18% vs baseline of 200)\n"
        f"New subscriptions activated: 3,955 (amounting to ${3955*19.99:,.2f} per month incremental)\n"
        f"Primary weakness: Delay in marketing cycle had a downstream impact on communications, leading to low awareness."
    )

@tool
def init_password_reset(email: str) -> str:
    """Initiate a secure password reset for a Peloton member account.
    Return a confirmation that the reset link has been sent to the customer/user's email."""

    # Simulated responses - in a real system, this would call an identity management API
    return(
        f"Password reset request has been sent to {email}.\n"
        f"You will receive a secure reset link in the email and it will expire in 30 minutes.\n"
        f"If the email is not received within 5 minutes, please check your spam/junk folder.\n"
        f"You can always request us to re-send a password reset link by visiting https://members.onepeloton.com/forgot-password/."
    )

@tool
def get_order_status(order_id:str) -> str:
    """Look up the current status of a Peloton order by order ID. 
    Returns the shipping status, the carrier, the tracking ID, and estimated delivery."""

    # Simulated responses - in a real system, this would call an order management API
    return(
        f"Order ID: {order_id} is in transit.\n"
        f"Shipped: November 11, 2025\n"
        f"Carrier: USPS\n"
        f"Tracking ID: AB123456789GB\n"
        f"Last scan: 2 days ago in Chandler, AZ 85224.\n"
        f"Estimated delivery date: November 28, 2025."
    )

# Registering the tools for each agent to align with Phase 1 architecture
agent_tools = {
    "business": [get_campaign_metrics],
    "membership": [init_password_reset],
    "order": [get_order_status]
}

# Binding tools to each agent
agent_models = {key: model.bind_tools(tools) for key, tools in agent_tools.items()}

print("Tools registered successfully.")
for key, tools in agent_tools.items():
    print(f"{key}: {[t.name for t in tools]}")

In [ ]:
# Defining the class
class PelotonState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    agent_type: Literal["business", "membership", "order"]

VALID_AGENTS = ["business", "membership", "order"]

# Router to classify user query to one of the agent types
def router_node(state: PelotonState) -> dict:
    latest_user = [m for m in state["messages"] if isinstance(m, HumanMessage)][-1]
    
    router_prompt = f"""Classify this Peloton customer/employee query into exactly one category. You must respond with ONLY one word: business, membership, or order.

    Categories are as follows:
    - business: marketing campaigns, ROI, brand performance, content strategy, competitor analysis, product positioning
    - membership: password resets, billing issues, subscription cancellations, refunds on subscriptions, account security, membership tiers, upgrade questions
    - order: physical product order status, shipping carriers and costs, delivery, warranty on products, returns of products

    Query: {latest_user.content}"""

    response = model.invoke([HumanMessage(content=router_prompt)])
    agent_type = response.content.strip().lower()
    if agent_type not in VALID_AGENTS: # this is our guardrail to not allow invalid agent invokes
        agent_type = "business" # fallback to default agent
    return {**state, "agent_type": agent_type}

def route_decision(state: PelotonState) -> str:
    return state["agent_type"]

# Defining a generic agent node builder
def make_agent_node(agent_key: str, agent_label: str):
    # Binding agents to their tools and helping the LLM understand when to invoke them
    bound_model = agent_models[agent_key]

    def agent_node(state: PelotonState) -> dict:
        latest_user = [m for m in state["messages"] if isinstance(m, HumanMessage)][-1]

        # Retrieving relevant context from the vectorstore using k
        retriever = agent_vectorstores[agent_key].as_retriever(search_kwargs={"k": 3})
        relevant_docs = retriever.invoke(latest_user.content)
        context = "\n\n".join([doc.page_content for doc in relevant_docs])

        system_prompt = f"""You are the {agent_label} for Peloton. Use the reference Q&A pairs below to answer the user's query in a helpful tone that is consistent with Peloton's brand. Consider the full conversation history when you respond to the user. Do not repeat prior information unless absolutely necessary. If reference material does not cover the query (partially or fully), please use your best judgment while staying on-brand.
        
        You also have access to internal Peloton tools. Use them ONLY when the user's request clearly requires a live-system lookup AND a tool matching that specific intent exists in your available tools. If the user's request does not match any of your available tools, respond conversationally without calling any tool. Do NOT invoke tools for general questions already covered by the reference material. 

        When a tool returns actionable details for the user (confirmation, expiry timelines, URLs, next steps etc.), you MUST include those details in your response so the user can reference it immediately without having to look elsewhere. 
        
        Reference material: {context}"""

        # Passing the system prompt and full history to the tool-bound model
        response = bound_model.invoke([SystemMessage(content = system_prompt)] + state["messages"])
        return {"messages": [response]}
    return agent_node

#### Agent 1: Business/Marketing AI agent
This agent can answer questions on marketing campaigns, content strategy, and other business topics

**Architecture:** RAG on knowledge base + tool call for campaign metrics

In [ ]:
business_agent_node = make_agent_node("business", "Business/Marketing AI Agent")

#### Agent 2: Membership/Fraud Detection AI Agent

This agent can answer questions on billing issues, fraudulent activity, and passwords

**Architecture:** RAG on knowledge base + tool call for password reset

In [ ]:
membership_agent_node = make_agent_node("membership", "Membership/Fraud Detection AI Agent")

#### Agent 3: Order/Shipping AI agent

This agent can answer questions on delays, delivery methods, warranty etc.

**Architecture:** RAG on knowledge base + tool call for order status

In [ ]:
order_agent_node = make_agent_node("order", "Order/Shipping AI Agent")

### LangGraph architecture

In [ ]:
# Setting up the nodes to connect the router to the 3 agents
graph = StateGraph(PelotonState)
graph.add_node("router", router_node)
graph.add_node("business", business_agent_node)
graph.add_node("membership", membership_agent_node)
graph.add_node("order", order_agent_node)

# Adding a shared tool node to hold the tools
all_tools = [get_campaign_metrics, init_password_reset, get_order_status]
graph.add_node("tools", ToolNode(all_tools))

# Routing on a conditional basis
def should_call_tools(state:PelotonState) -> str:
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return END

# Routing the run to the same agent to synthesize final answer
def route_back_to_agent(state: PelotonState) -> str:
    return state['agent_type']

# Wiring the final graph
graph.add_edge(START, "router")
graph.add_conditional_edges("router", route_decision, {
    "business": "business",
    "membership": "membership",
    "order": "order",
})

# Each agent has to have either tools branch or END
for agent in ["business", "membership", "order"]:
    graph.add_conditional_edges(agent, should_call_tools, {
        "tools": "tools",
        END: END,
    })

# Routing back to agent
graph.add_conditional_edges("tools", route_back_to_agent, {
    "business": "business",
    "membership": "membership",
    "order": "order",
})

# Compiling the graph
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# Displaying the graph as a mermaid diagram
display(Image(app.get_graph().draw_mermaid_png()))
print(app.get_graph().draw_mermaid())

### Defining the 9 multi-turn user stories

In [ ]:
# Turn 1 (test_query) is derived from the testing rows (Phase 2) to ensure continuity
# Turns 2-3 will probe deeper into training rows' content
# `architecture` field will track if the design aligns with what I detailed in Phase 1
multi_turn_stories = [
    # BUSINESS/MARKETING
    {
        "story_id": "B1",
        "expected_agent": "business",
        "user_story": "As a marketing manager, I want an AI agent to help analyze the success metrics for the Summer Fitness Challenge",
        "architecture": "Tool call (get_campaign_metrics) + RAG context",
        "turns": [
            "Can you pull the metrics for the Summer Fitness Challenge? I need to understand why it underperformed.",
            "That participation rate is concerning. What KPIs should we watch more closely to prevent that outcome from recurring next year?",
            "Given the 75% participation benchmark, what's a target we should aim for?",
        ], 
    },
    {
        "story_id": "B2",
        "expected_agent": "business",
        "user_story": "As a content strategist, I want an AI agent to help recommend blog topics based on trending workouts and member engagement.",
        "architecture": "Pure RAG",
        "turns": [
            "Suggest a blog topic for our least engaged audience sections.",
            "What about topics that would resonate with members who mostly take Yoga classes?",
            "Which of those two directions would work best for a June launch?",
        ],
    },
    {
        "story_id": "B3",
        "expected_agent": "business",
        "user_story": "As a product manager, I want to compare Peloton's market share against our competitors.",
        "architecture": "Pure RAG",
        "turns": [
            "How did Peloton's market share evolve in this year versus previous years?",
            "What are SoulCycle and Echelon doing right now that we aren't? Is it something to watch more closely?",
            "Given these dynamics, what is your recommended positions for our next product?",
        ],
    },
    # MEMBERSHIP/FRAUD DETECTION
    {
        "story_id": "M1",
        "expected_agent": "membership",
        "user_story": "As a member, I want the AI agent to help reset my account password.",
        "architecture": "Tool call (init_password_reset) + RAG Context",
        "turns": [
            "I need to reset my password. My email is member@peloton.com",
            "My reset email didn't arrive. I've already checked my spam folder and I don't see anything.",
            "Once I get the reset email, what should I do to keep my account more secure?",
        ],
    },
    {
        "story_id": "M2",
        "expected_agent": "membership",
        "user_story": "As a member, I want the AI agent to assist me in resolving a billing issue with my subscription.",
        "architecture": "Pure RAG",
        "turns": [
            "I received a charge for a subscription I had cancelled.",
            "My cancellation was on July 15. What would my subscription status have been on that date?",
            "If the cancellation wasn't processed correctly, what are your next steps? I think a refund is in order.",
        ],
    },
    {
        "story_id": "M3",
        "expected_agent": "membership",
        "user_story": "As a member, I want the AI agent to explain the benefits of moving from a basic membership to a premium tier",
        "architecture": "Pure RAG",
        "turns": [
            "Are there any hidden fees with upgrading to a premium tier?",
            "What's included with premium beyond the classes?",
            "Do you have a free trial before I commit fully?",
        ],
    },
    # ORDER/SHIPPING
    {
        "story_id": "O1",
        "expected_agent": "order",
        "user_story": "As a customer, I want the AI agent to investigate the status of my order.",
        "architecture": "Tool call (get_order_status) + RAG Context",
        "turns": [
            "My order hasn't arrived yet. Can you check the status? My order ID is 'AB123456789GB'.",
            "It's been over 10 days now. That seems unusually long for an order to arrive?",
            "If it still hasn't arrived by next week, what are my options?",
        ],
    },
    {
        "story_id": "O2",
        "expected_agent": "order",
        "user_story": "As a customer, I want the AI agent to clarify the warranty policy on the new Peloton bike.",
        "architecture": "Pure RAG",
        "turns": [
            "Is there a warranty if I buy a used Peloton bike?",
            "What if I bought it from a certified reseller?",
            "Do the accessories that came with the used bike have any warranty coverage?",
        ],
    },
    {
        "story_id": "O3",
        "expected_agent": "order",
        "user_story": "As a new employee in the shipping department, I want the AI agent to provide me with more details about our shipping partners and costs",
        "architecture": "Pure RAG",
        "turns": [
            "How do we handle international shipping for our products?",
            "Which of our shipping partners handles European deliveries?",
            "What are the cost differences for express vs. standard international shipping?",
        ],
    },
]

print(f"Successfully defined {len(multi_turn_stories)} multi-turn user stories.")
print()
print("The architecture distribution:")

arch_counts = Counter([s["architecture"] for s in multi_turn_stories])
for arch, count in arch_counts.items():
    print(f"- {arch}: {count}")

### Running the user stories

Each user story will be run as an independent conversation with a unique `thread_id` (reusing a `thread_id` can cause checkpoint memory bleed). For each story, I will generate a new `thread_id`, invoke turns sequentially and make the graph read the accumulated messages, then ensure the LLM invokes the tool where needed. 

The full conversation (user queries, agent response, tool calls) will be rendered inline. 

In [ ]:
def run_multi_turn_story(story):
    """Run a single multi-turn conversation and render it inline."""
    # Generating a unique thread ID to isolate this conversation
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    # Story header (NO leading indentation - or markdown treats it as a code block)
    header = (
        f"---\n\n"
        f"### Story {story['story_id']}: {story['expected_agent'].upper()} agent\n\n"
        f"**User Story:** {story['user_story']}\n\n"
        f"**Architecture:** {story['architecture']}\n\n"
        f"**Thread ID:** `{thread_id[:8]}`\n"
    )
    display(Markdown(header))

    # Tracking message count so tool calls are rendered from the CURRENT turn. I will accumulate (state["messages"] across turns via MemorySaver
    prior_msg_count = 0
    result = None

    tool_names = {"get_campaign_metrics", "init_password_reset", "get_order_status"}

    for turn_idx, user_input in enumerate(story["turns"], start=1):
        result = app.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
        )
        final_msg = result["messages"][-1]
        routed_to = result["agent_type"]

        # Agent should look only at messages added during this turn
        new_msgs = result["messages"][prior_msg_count:]
        turn_tool_msgs = [
            m for m in new_msgs
            if getattr(m, "name", None) in tool_names
        ]
        prior_msg_count = len(result["messages"])

        # Building the turn markdown
        turn_md = f"**Turn {turn_idx} - User:** {user_input}\n\n"

        for tool_msg in turn_tool_msgs:
            turn_md += (
                f"**Tool `{tool_msg.name}` invoked, returned:**\n\n"
                f"```\n{tool_msg.content}\n```\n\n"
            )

        turn_md += f"**Turn {turn_idx} - Agent ({routed_to}):** {final_msg.content}\n"
        display(Markdown(turn_md))

    return result

In [ ]:
# Running all stories
print(f"Running {len(multi_turn_stories)} multi-turn user stories:\n")
demo_results = {}

for story in multi_turn_stories:
    demo_results[story["story_id"]] = run_multi_turn_story(story)

print(f"\nAll {len(multi_turn_stories)} user stories executed.")

#### Findings

- **Phase 1 alignment on architecture:** My design above implements the architecture I prescribed in Phase 1. Three of the nine stories, i.e., Summer Fitness Challenge metrics (`get_campaign_metrics`), password reset (`init_password_reset`), delayed order status (`get_order_status`), use tool calls and RAG context in conjunction. The other six user stories use a pure RAG architecture. I had previously opined that campaign analytics, user identity actions, and order retrievals would need deterministic actions rather than AI reasoning as they operate at a higher standard. The tool bindings I used have simulated data/actions but would need active connections to Peloton's systems in a production environment (e.g., an MCP server to fetch order status).

- **Multi-turn discipline:** All nine conversations rely on LangGraph's `MemorySaver` and I generated a unique `thread_id` for each story. This is a conscious design choice to prevent memory from previous conversations from bleeding over to current ones. I also used `Annotated[List[BaseMessage], add_messages]` to handle automatic message accumulation. This lets the LLM rely on prior context when my flow invokes Turns 2 and 3.

- **Tool re-invocation:** One observation is that Story M1 (password reset) invokes `init_password_reset` in both Turn 1 and Turn 2. While it is expected behavior in Turn 1, I believe the LLM read the user saying the email did not arrive as an implicit request to send the link again and appease the user. I am treating this as expected behavior as emailing a user carries a negligible cost versus not complying with their implicit requests, if any. In a production setting, the design should: (1) have a diagnostic to check if password reset was sent (e.g., "We already sent a reset link to your email in the last 15 minutes. Would you like us to resend it?"); and (2) a rate limiter to prevent repeated invocations and to make sure the previous link was deactivated before a new one is sent (e.g., for a distinct user, no more than 4 password resets can be requested in a 24-hour period). 

- **Router boundary:** In my runs so far, the router has successfully classified each story to the appropriate agent. There was some instability when my questions had keywords that could have fallen into two or more categories but even those cases were appropriate (e.g., asking what is the policy around shipping can be a business question or a shipping question depending on user intent). I rephrased the test queries in those cases to remove ambiguous trigger words and I believe it is a valid design choice, as we are trying to test the LLM's ability to classify semantically. 

**Conclusion:** This exercise was extremely helpful to understand how to design and manage multi-turn conversations. Through this project, I have obtained a much deeper understanding on how to make AI agents more effective at their tasks. Specifically, I learned how to wire tool calls and RAG context within the same agent and retain the state across conversation turns. As I mentioned before, a production environment would demand tool calls to live systems but the graph structure here is more than capable of handling these user stories. 